In [9]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
# logging.langsmith("CH03-OutputParser")

llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini"
)

In [10]:
email_conversation = """
From: 백효영 (gydud7300@naver.com)
To: 조지러셀 (russell7300@naver.com)
Subject: "2026 스페인 그랑프리의 타이어 전략 미스에 대한 이야기"

안녕 조지러셀아. 
다름이 아니라 너 이번 스페인 그랑프리때 팀이 너의 타이어 전략을 이상하게 세운거 같은데 너는 어떻게 타이어 전략을 가져가야했다고 생각하니?

백효영
메르세데스 팬
자택경비원
"""

In [11]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음 이메일 내용의 요지를 추출해 주세요.\n\n{email_conversation}"
)

llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
)

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

이메일의 요지는 백효영이 조지러셀에게 2026 스페인 그랑프리에서의 타이어 전략에 대한 문제를 언급하며, 그에 대한 조지러셀의 의견을 묻고 있다는 것입니다.

In [12]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 받는 사람")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")

parser = PydanticOutputParser(pydantic_object=EmailSummary)

print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 받는 사람", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}}, "required": ["person", "email", "subject", "summary"]}
```


In [13]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
""")

# format 에 PydanticOutputParser의 부분 포맷팅(partial) 추가
prompt = prompt.partial(format=parser.get_format_instructions())

# chain 을 생성합니다.
chain = prompt | llm


# chain 을 실행하고 결과를 출력합니다.
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 JSON 형태로 출력됩니다.
output = stream_response(response, return_output=True)


# PydanticOutputParser 를 사용하여 결과를 파싱합니다.
structured_output = parser.parse(output)
print(structured_output)
structured_output.person


chain = prompt | llm | parser


# chain 을 실행하고 결과를 출력합니다.
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 EmailSummary 객체 형태로 출력됩니다.
response

```json
{
  "person": "백효영",
  "email": "조지러셀",
  "subject": "2026 스페인 그랑프리의 타이어 전략 미스에 대한 이야기",
  "summary": "스페인 그랑프리에서 팀의 타이어 전략이 이상하게 세워졌다는 내용과 조지러셀의 의견을 묻는 질문."
}
```person='백효영' email='조지러셀' subject='2026 스페인 그랑프리의 타이어 전략 미스에 대한 이야기' summary='스페인 그랑프리에서 팀의 타이어 전략이 이상하게 세워졌다는 내용과 조지러셀의 의견을 묻는 질문.'


EmailSummary(person='백효영', email='조지러셀', subject='2026 스페인 그랑프리의 타이어 전략 미스에 대한 이야기', summary='스페인 그랑프리에서 팀의 타이어 전략이 이상하게 세워졌다는 내용과 조지러셀의 의견을 묻는 질문.')